# 2 · LLM Attack Generator

**Makes the held-out test attacks.** Two local LLMs write SQLi/XSS payloads your detector has
**never seen** — the evasion test set. These are **test data only** and are never used for training
(Section 3.5).

Second of three notebooks:

| # | Notebook | Makes | Needs |
|---|---|---|---|
| 1 | `honeypot_generator` | the training corpus | 5 WEB-IDS23 CSVs |
| **2** | **llm_attack_generator** ← you are here | the held-out test attacks | **GPU + Ollama** |
| 3 | `end_to_end_pipeline` | trained models + all results | outputs of 1 and 2 |

---

## Files you need

**Almost nothing** — the two generator scripts and the validator are embedded here. You need:

- a **GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU)
- **Ollama**, which Step 2 installs for you
- ~9 GB of disk for the two model pulls

Outputs land in `data/eval/`:

| File | Contents |
|---|---|
| `codellama_holdout.csv` / `deepseek_holdout.csv` | accepted payloads — the frozen test corpus |
| `*_rejects.csv` | rejected payloads **with the reason** — your audit trail |
| `*_run_manifest.json` | model tag + digest, decoding params, counts |

---

## The two models

| Model | Ollama tag | Keep |
|---|---|---|
| Code Llama 13B | `codellama:13b` | 500 SQLi + 500 XSS |
| DeepSeek-R1 14B | `huihui_ai/deepseek-r1-abliterated:14b-qwen-distill` | 300 SQLi + 300 XSS |

**Total kept: 1,600 payloads.**

DeepSeek is the *abliterated* build — the aligned release refuses to write payloads. The namespace
underscore (`huihui_ai`) is the Ollama registry name; the HuggingFace `huihui-ai` (hyphen) is **not**
pullable (no GGUF → 400).

---

## Why we ask for far more than we keep

Payloads are thrown away for three reasons:

1. **validation failure** — a refusal, reasoning prose, or no attack signature
2. **duplicates** — deduped on a normalised key (NFKC + casefold + collapsed spaces), so
   near-copies count as one
3. **malformed output** — the model ignored the JSON format

So we over-request. The generator asks in **batches of 20** and **stops the instant a target is
reached**, so extra headroom is free when yield is good and rescues the run when it is not.

| Model | Keep/type | Observed accept rate | Raw requested | Batches |
|---|---|---|---|---|
| Code Llama | 500 | 78.4 % | 960 | **48** |
| DeepSeek | 300 | 10.1 % | 4,460 | **223** |

> DeepSeek's rate falls **below** 10.1 % once reasoning prose is correctly rejected — the 1.5×
> safety margin absorbs that. The old hard-coded 15/25 batches were both too small; that is the
> mechanical reason the earlier smoke test came up short.

**Cost:** the DeepSeek run is thousands of requests and can take hours even on a T4. **Smoke-test
first** with the small numbers, confirm the checkpoint, then commit to the full run.

---
## Step 1 — Harness

In [ ]:
# ==========================================================================
# HARNESS - logging / hashing / timing used by every step below.
# Touches no data. Safe to re-run.
# ==========================================================================
import os, sys, json, time, hashlib, platform, subprocess, traceback, shutil, textwrap
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager

# Attack payloads contain characters outside cp1252 (full-width quotes, CJK).
# Without this, printing one raises UnicodeEncodeError and kills a healthy run.
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass
os.environ.setdefault("PYTHONIOENCODING", "utf-8")

RUN_ID      = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
RUN_STARTED = time.time()
RUN_LOG     = []
_STAGE_SEQ  = [0]
IS_COLAB    = ("google.colab" in sys.modules) or os.path.isdir("/content")
RUN_LOG_PATH = Path("/content" if IS_COLAB else ".") / ("aigis_%s.jsonl" % RUN_ID)

_W = 78
def rule(ch="="): print(ch * _W)

def fmt_bytes(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB"):
        if n < 1024.0 or u == "GB":
            return ("%d %s" % (n, u)) if u == "B" else ("%.1f %s" % (n, u))
        n /= 1024.0

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def describe(path, label=None):
    p = Path(path); label = label or str(p)
    if not p.exists():
        return {"label": label, "path": str(p), "exists": False}
    st = p.stat()
    if p.is_dir():
        fs = sorted(x for x in p.rglob("*") if x.is_file())
        return {"label": label, "path": str(p), "exists": True, "is_dir": True,
                "n_files": len(fs), "bytes": sum(x.stat().st_size for x in fs)}
    return {"label": label, "path": str(p), "exists": True, "is_dir": False,
            "bytes": st.st_size, "sha256": sha256_file(p),
            "mtime": datetime.fromtimestamp(st.st_mtime, timezone.utc).isoformat()}

def _pd(d, pre="   "):
    if not d["exists"]:
        print("%s[MISSING] %s  ->  %s" % (pre, d["label"], d["path"]))
    elif d.get("is_dir"):
        print("%s[dir ] %s: %d files, %s" % (pre, d["label"], d["n_files"], fmt_bytes(d["bytes"])))
    else:
        print("%s[file] %s: %s  sha256=%s...  mtime=%sZ"
              % (pre, d["label"], fmt_bytes(d["bytes"]), d["sha256"][:16], d["mtime"][:19]))

_CURRENT = {"rec": None}

def check(cond, msg, fatal=False):
    """Recorded assertion. A step that 'ran fine' but produced nothing must fail loudly."""
    ok = bool(cond)
    print("   [%s] %s" % ("PASS" if ok else "FAIL", msg))
    if _CURRENT["rec"] is not None:
        _CURRENT["rec"]["checks"].append({"ok": ok, "msg": msg})
    if not ok and fatal:
        raise AssertionError(msg)
    return ok

def sh(cmd, cwd=None, env=None, timeout=None, echo=True):
    """Run a command, streaming output live AND capturing it to the run log."""
    if isinstance(cmd, str):
        shown, kw = cmd, {"args": cmd, "shell": True}
    else:
        shown, kw = " ".join(str(c) for c in cmd), {"args": [str(c) for c in cmd], "shell": False}
    if echo: print("   $ %s" % shown)
    t0, lines = time.time(), []
    pr = subprocess.Popen(stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=str(cwd) if cwd else None,
                          env=env, text=True, bufsize=1, errors="replace", **kw)
    try:
        for line in pr.stdout:
            line = line.rstrip("\n"); lines.append(line); print("   | %s" % line)
        pr.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        pr.kill(); lines.append("*** TIMEOUT ***"); print("   | *** TIMEOUT after %ss ***" % timeout)
    print("   -> exit=%s in %.1fs (%d lines)" % (pr.returncode, time.time() - t0, len(lines)))
    if _CURRENT["rec"] is not None:
        _CURRENT["rec"]["commands"].append({"cmd": shown, "returncode": pr.returncode,
                                           "seconds": round(time.time() - t0, 2),
                                           "output_tail": lines[-40:]})
    return pr.returncode, lines

@contextmanager
def stage(name, purpose, inputs=None, outputs=None, params=None, optional=False):
    _STAGE_SEQ[0] += 1; n = _STAGE_SEQ[0]
    rec = {"run_id": RUN_ID, "stage_seq": n, "name": name, "purpose": purpose,
           "params": params or {}, "optional": optional,
           "started_utc": datetime.now(timezone.utc).isoformat(),
           "commands": [], "checks": [], "notes": [], "inputs": [], "outputs": [],
           "status": "running"}
    prev = _CURRENT["rec"]; _CURRENT["rec"] = rec
    rule("="); print("STEP %d: %s" % (n, name)); rule("=")
    print("PURPOSE : %s" % purpose)
    if params:
        print("PARAMS  :")
        for k, v in params.items(): print("   %s = %r" % (k, v))
    if inputs:
        print("INPUTS  :")
        for l, p in inputs.items():
            d = describe(p, l); rec["inputs"].append(d); _pd(d)
    print("START   : %sZ" % rec["started_utc"][:19]); rule("-")
    t0 = time.time()
    try:
        yield rec
        if rec["status"] == "running": rec["status"] = "ok"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = {"type": type(e).__name__, "message": str(e),
                        "traceback": traceback.format_exc()}
        rule("-"); print("!! STEP %d FAILED: %s: %s" % (n, type(e).__name__, e))
        print(textwrap.indent(traceback.format_exc(), "   "))
        if not optional:
            _finish(rec, t0, outputs, prev); raise
    _finish(rec, t0, outputs, prev)

def _finish(rec, t0, outputs, prev):
    rec["seconds"] = round(time.time() - t0, 2)
    rule("-")
    if outputs:
        print("OUTPUTS :")
        for l, p in outputs.items():
            d = describe(p, l); rec["outputs"].append(d); _pd(d)
    nf = sum(1 for c in rec["checks"] if not c["ok"])
    v = rec["status"].upper()
    if rec["status"] == "ok" and nf: v = "OK (with %d FAILED check%s)" % (nf, "s" if nf > 1 else "")
    print("RESULT  : %s   duration=%ss   checks=%d passed=%d failed=%d"
          % (v, rec["seconds"], len(rec["checks"]), len(rec["checks"]) - nf, nf))
    rule("="); print()
    RUN_LOG.append(rec); _CURRENT["rec"] = prev
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
    except Exception as e:
        print("   [warn] run-log append failed: %s" % e)

def note(msg):
    print("   * %s" % msg)
    if _CURRENT["rec"] is not None: _CURRENT["rec"]["notes"].append(msg)

def skip(reason, produced_by=None):
    print("   [SKIP] %s" % reason)
    if produced_by: print("          -> produced by: %s" % produced_by)
    if _CURRENT["rec"] is not None:
        _CURRENT["rec"]["status"] = "skipped"
        _CURRENT["rec"]["notes"].append("SKIPPED: " + reason)

def preview_json(path, max_chars=1400):
    p = Path(path)
    if not p.exists(): print("   [MISSING] %s" % p); return None
    obj = json.loads(p.read_text(encoding="utf-8"))
    txt = json.dumps(obj, indent=2, ensure_ascii=True)
    print(textwrap.indent(txt[:max_chars], "   "))
    if len(txt) > max_chars: print("   ... (%d more chars)" % (len(txt) - max_chars))
    return obj

def checkpoint(title, fields):
    """Compact copy-pasteable block for sharing run state."""
    print()
    print("+" + "-" * 66 + "+")
    print("| %-64s |" % (title + " - copy between the +---+ lines"))
    print("+" + "-" * 66 + "+")
    for k, v in fields.items():
        print("%-15s: %s" % (k, v))
    fails = [c["msg"] for r in RUN_LOG for c in r["checks"] if not c["ok"]]
    print("%-15s: %d" % ("failed checks", len(fails)))
    for f in fails: print("   - %s" % f[:76])
    print("+" + "-" * 66 + "+")

print("Harness ready.  RUN_ID=%s   env=%s" % (RUN_ID, "Colab" if IS_COLAB else "local"))

---
## Step 2 — Set up: locate project, install Ollama, unpack the generators

In [ ]:
# ========================================================================
# EMBEDDED GENERATORS - this notebook carries its own code.
#
# Each file below is gzip+base64 embedded here and written to scripts/
# at runtime. Nothing is downloaded; no external .py is required.
#
# Written to disk rather than exec()'d inline because the scripts import
# each other and several are launched as subprocesses (so a crash cannot
# kill the kernel, and each stage can capture its own stdout).
#
# A file is overwritten ONLY when its content differs, and every such
# overwrite is reported - a local edit is never silently clobbered.
#
# 4 files, 38 KB raw
# ========================================================================
import base64, gzip, os
from pathlib import Path

_EMBEDDED = {
    "25_generate_codellama_corpus.py":
        "H4sIAJ4HjWoC/81a63LbyLH+j6eYwJUSEJMQKdupU0gxVbJFr5WVJUWU9pyNVoUaAkMRKxDA4iKJq+Kp8xB5wjxJvp4LLiTt"
        "3eTsj7BsETOY6enu6f66e4avfndYl8XhPE4PRfrI8nW1zNI3lm3b1tG74F6kouCVCMIsEknCVxxPRV6XXr5mjDkfljyvRMHe"
        "DNhMhFWcpeyN944Nh2zOS5HEqWCaRFa4lnVZZFEdipJVS8GWIomGWV2xD6DNzog4G795z3j0KIqSFzFPWFnPS1HJ4fhfxiUr"
        "cxHGi1iUvvW/70YjNvvrWcxeM/n8P7MZS7NHkbCcr5OMR+WAxSk7egduqhDzWbZgRyOWg2NeVTx8YNU6FwPrKa6WcpEoC+uV"
        "SCsRsVLgzyPYwBdNkCTYIitYmKVVkSUJXkQx8RpXa8+yrpdgL1ayvb85+3bAFnwVJ+thWRV1WNUFxjfK8NhpRYPPL67lhKM3"
        "Q82zRWvytGKFeCpiUq4zHrcbIR55CTW32+B67DyDniEaqUg0soO8JcAeq0usTIxXBY/TOL0HZ4ITQxAyUds2YI9ZyOd1wov1"
        "gFW1GlaSjuRTVljVshDlMkuidhZtNBZdM1509rMSZcUiXnGWpQlp5v3xh2+n5yc+4ym7kFYEEgVY89hlFkPUuMJ+0Nvjn4mr"
        "by5v2Hef2XzNwiVP78EAFlpmZWU52BV2/Lebq2lwdXP+/uLiW28VuX+SKjy+PCWNxhH2Lw5hPFXGEgiV0Exw8eHi/Prq4uxs"
        "esK+mZ5Pr46vTy/OmdOxW9cH76ucFA02JiMoIsuDh8mYOSKFAkPocb62ODEZZNodkhjreUqsD7LhDvYbz4CJZx5WbEW+xCp+"
        "D4kjK4rvSV+FwI5GGA59ZFKeok7ZiqfxQkj2L26uL2+uZ8yp0wgUSb+HMIbk0PUtxlr/pB3CLnhh+ciaDw9DkZNVN8bhLIrs"
        "Z5GqzVLG5PboFOJHaKbs01GdXTqv0cdL6M+RFga+w6yW1ptnRbVFsk6DRqIfaRI+jTYOlSaU8uC50NtQeZAiWQ4s9ksfJSdP"
        "Q6gP3oINjFegyVf5AHsQPog0YtKQrJvZ8TdTqYUVdFku5xkvIqXLV8ps8KTQkJVhEedVefhLgDgcKmGad/74zVwSVHYNm87I"
        "IWHos9kn8rIUo53GSP3x+O2bt2z4Z4xUz+5vxIT2n9YfJFfligPFCnxnD8re5gK7SFu4WsVVRY5HvrjM6qL87TgxYHwko4zc"
        "gSBY1ORzQcDiFRkOdJRmFSfPLC3L9BX3OS9KYdowTfNI1mSey3VpHmn7Lb3HlaCWoW/aykR+BoKqcTmvlkk8N8Mu0bQsUPTo"
        "hRenAK7KGRE0Fg69dMB6nIBxFyMK8n5XEdqDEoZoFyxAacmP3v0xyBaSaJwHC5FSjKQdSrOfuM+mb0dHhjvpdgH8Po6kdgzN"
        "MBGcUFy9gFhpVmB7459FFDyIdesiPaLW1QUC0ITtlUR/EfJgiBx5yGwCHpseCHxseumtHqK4cNTocnJd1IKgLi6rIHuQTZdi"
        "wGwazKYA3wl7e9T12Ve91GErAv9pNwTPRZI9WSfTj8c3Z9fB++PrD5+mM1A9emfJRjA7/duU2iPL6tH2mS2zBW201PsUJ2Sf"
        "5JTzLCMVDilxAcwjd4kG7OYcMUJ1DUCMTKU/QBRFVugBiIDPPEJ6Ar2zbL6oy5Cr2AqcV7FURMOfalGsiZYIl2mMVunJxGU/"
        "V4VYJBJxyTjgmvg+ufhsFqT48YpSmSKe15VhDZaKtCEtPduCvKfBx+PPp2enUke3thYzkPzbA2Yj1COdkDOpSSKqlhmyi7q2"
        "lLqdA7mDjrzUpaUNpLT2nQUJ+3w0gqnRJBo9RdmqpdsIprvuLGv2/ex6+jm4vLr4fElW6Uju7O+zWuYglFiENdKmNTRXCl4A"
        "aLC1dZxEBGa8TVL++/jj4enJjJEV10rttqKlAEwmTJzBzKbns9Pvpg09lhcZBUHKLAHkcZklnKw1WyxkvpvwuceIH00uVzkv"
        "K8FAIv2SElSdfjZhVOZolC5R1iJjMmeRgI1UMqoiO9LkkOHwBCE3Wqt0TkQeu6irHCIVdYK0+IdUjxw3Ly7Oz74Hvb/MkO/w"
        "ouBrShYJa9J7xFnBIZRqMUokC/5k+PIaYkeUZSIdKR6i7CklKWSQgKPKhBHNtF7NRaFytd5SlAM2dN54bErr6QXYqoaoc9q4"
        "RZ1KT4XvwI1WeWJSWmievIR03PhMS/Ctx74DC6xJspEDFFmp0nDJgY8sfbEQBE4MSPiENAtSlzkPZWYLsM0i+aQpkrP2vLfM"
        "SL7qKWu3i2wthTkMm2yz5eidx05oBllImNSobcRznvBUBTN0lpgk2TPk4KhASOUf3wfvr06nH2HaL4pc32H9pqOLQ1QFgfCP"
        "Gusc6EIcLniCaiBOpeghQbJ377Hj8xM2RkZ7qJ+OoLl1Ilzt5z04wGIdANxaRBcB5FXSlpGthcgskFKGWVKv0tJQ3EEUX/V9"
        "RYCanIWViRD5IXIHWJGxjoZqF4J83dzLJ3RFJYwqSB5jLh9ooMLuXcrbaKbCxg53HbAfShuSGX8CS5Vwv4qf0WGMy9Duw6Lf"
        "dKiosMU5QHwVk9lox8RYlCdkxrQA0rVUah+WBpAzK7S46ncaMsLAeTJSOAGPAq9P19eXBGw5xBctjxKMffMk5xp/RXWCHJpc"
        "MaOQjIpccbOI4REEmQSGBfSMojgrGp22sA6yTQBTXD0DsaWMKDCz+n7JVLI0lG6CsUyWwQ2p7aDgd7qGrfKI9LzQe0+ADzCR"
        "En8+awMmk4U/ulEnQ60oNqNEqnJjWVYkFipyBIDwVV45ys4DOjLwCW9Mga8bDz5px6X0HW3fUvUSICnVUYo+C/sbnTKzl4cN"
        "kKmE6AgmL4qUh7Ip4aFwDoKDATtgB+6mRQmDMIrOtcFCn730oONWkbrbbE04bk88MKUjjFfn2ErH3Z7wtehBzHcChYkmvprv"
        "avXJbD2QkxyMloqSCkog9y0ad1pNoDTpZb803LVUenit4D2GmjpMwOKoLqYhVbH2G77hg6BF9YAn1dZSok+8QMQGBMsa0cHY"
        "geTF9XtZjt424tB5dqVdP5PLYPydHCieqc5UqxBPJ4IAYEqA0lLKeVlqET6i0CK/gwYy1CCyLiqZQJElPdtokZJcSiI8NuPJ"
        "I7+XYWJFAYhrQkAbcLOC7zwVnLYNCTMQh6LNE0eMlHHlaYkIobLlg1JWcIS3SldEvqQMLPGkvh1Xf9sD2zwqiROSGMrzyjyJ"
        "KzkPb6DAZqbSRalYjYio6mlmy0l+V/my27FvKb27s7fUTgdrcVqL3gSROonL/ox8XgJNcju6Y5MJvofju7YHVO2DH2y7T5As"
        "Ibkd+xja9Bt2PVJfGoF411PNW23ATX0LkaSvOKGu2uQWDljHjTQcxKIcmPNGFJeoWYU600i1tOYsZtCeppDmBo32+CqI0TVq"
        "dDlXO5HeC0cTdv1tHRmqUlVq1b4uJBi23q0OViYNz7dq2d9LYqbTvetOwOvXEzZuNSkk621l9xqc7vu86lZ1dKyjCjl50Elg"
        "3XqMBFrQ/BLuGsgdsLbUczvTAcDOwmbstotvmztdOb7MX483hy9ahxujhYkG340SafJCfzd2S7eHLy1e6aM/YyOONgnF9YD1"
        "ShV1qtWS1AByVafkmhI4GAci9NdpBWLsRXRZ6h7GGUt2bHufQa7V2nC4tsQMFhylfmS77tcdUB53k5PBALfAfMt3c6kQ5KUO"
        "je+TzR4G5pRw0pxQOHmP2f4MGDVBWvbg71SfO1LnXxVarbsl515Z6UMnJZOtoxMn3+GNhsWpdOr/N4N2VANfkWsK+9dySet6"
        "PIoc8NGfYlDgV63t7sj1q4CkDyYaN3dBTSPoCjWqY5CPjMQc4nnHxb3EhEtqFU4k1LkirHMSBACMIPgVB74UDOVxRxAmCLeT"
        "hvoVfzppKX4SSf7RDHU1M6RC2LPiwrH1cSV8CHzzOqkmdu/k0v7iPDpQ7U5rTlnt/RIswc3E1jch8naC3aAeaaY1aWnnlFhf"
        "CpiDZO/L3NBxILih/Z7EFKoMX82J8t5pGhX3ztw6bPsiDWUuQ4J4IrKX1rvR6IvzeV0th1X2INKOOul266t6zPWxwRwFORIi"
        "OZ+smfT13We6E5qLJRWZ0CitoCp2WXIAq5/XjS6Le8qMwJeBuntkPJZ8pw9wJ72zW4c2i0yu9OgJ0A/VqzY9DeRqgeRH9bZt"
        "k3SEbRyB9uGm0gYdOVo+6vWrEQbKLI7+OGq6xB/8Vak0ZdCi0q9M5Pgh1fUGSStPPx21SagAaA3VCLBjKlS6zPOMPsqfkjgg"
        "7JYPc07BfjchanbGZEYt73Syh6nYy945pB7S5Ej7GNEZU0+WjiRU3f0LkjyXpRSEvv89OTATYnRPMf8VKXp5H+2T0iwSJsWZ"
        "1btbm7QKf21YtnT+/4+//x/+qZK4paiPLHt3enTM4erxv/k/5RNqVfjEjbwZ2HsHqQpCCWgZQpKjJkGjT9BnKp6oRJhQ4mKO"
        "SiZ2XS2G/2W7lA8t2tAjM67y0VOX4s6iDV5Pui97cm5tXUrp42MT+DpNFQHlqbMQ0RcQ2k74XIYCu7mxp4a0hqDi99QwdzHq"
        "/sa+c3tZkwy8JuTqC+G4jZP9kNqVYM/EARsPOuq1vxwXuzbb3CshgbnTNggb27dfnbve7f3Cq/+gzaIjLWI1UGldV+kt4dIx"
        "ruRuuU33Sr138w+MEfSTBHURQAfOcSGP2v5dD5LrUv5DdZO6vnaItU7WHNLBdCixm36mUhGAv2x6RhQM6J+0g0DWgKCwVTJT"
        "3UYzQ9QhlSOHjlzgRlum6QwtVLpoxDcHzNLcWyv39xtax/T9rpVtD1B3+BiD0CY5snUPts52O8P1bXwg0ySfLeyXJo5u/Jcm"
        "hm7s3hxEZWkIflt19t5LLG7g17BqMHprZFAixZekTCnZGdH+yCdoyyYMfrE7PxFBe+SN6NaMfieC1lg/5/TsjTYdgluhwfC2"
        "HTHaCQYq5KIyjvoyQdfBw9VRSXWqOOLK5Sue6N4ml+8yYpwj6I5sPGaHATooCyhU0qbCkCOnXyUcwqSet/peb9FkY/x/26Wt"
        "HFt7RlfCjr/sCNp5p+XtyiUSntMlQymwd1HZsNvJmtgQiZTkZsf0MbGuQkwyPwzwUkCU+W2Ah3euF5eZqjecLgEUqqQb+5O5"
        "WpQ3Xuz8S7+9Mj+ukhd7Oz+p8rTBb7ZweBezt39MY2uc/iVsloeWUb3KHTMdEEMIRFdZkyONmSrpQv5oYzftic3+wP74tp+P"
        "HZtYxl76xobUTKaaL11z3YB9StteOua6ce0+ySttMppkY0GbnZ/2sBfD/e3BlqEe3P1hPBr53nix+f0W/Q8qZWGgrxKRzdaA"
        "z5ooBiiNH3xF4wf92a2WLAvlRxCkfEW/aJlMmB0EVAgHgT6gVFWx9U/1CY4OfSkAAA=="
    ,
    "26_generate_deepseek_corpus.py":
        "H4sIAJ4HjWoC/81b63LbSHb+z6foYGrL4IgESVn2Tuhwt2SJHisjS44o7WRWVqFAoiliBAIwLroMw1QeIk+YJ8l3TnfjQlK+"
        "VCWpyLIINLpPn/sNzR/+oVdkaW8aRD0Z3YvkKV/E0cuWZVmt/dfurYxk6uXS9aVMMinv3FmcJkXmJE9CCPto4SW5TMXLjpjI"
        "WR7EkXjpvBLdrvB8PAnupdAQ4rTdan1MY7+YyUzkCykWMvS7cZGLY4CeAHT3YiAGB2+x9F6mmZcGXiiyYprJfCj+/WW/Lyb/"
        "chqIPXX9r5NJK/GewtjzM3EfeAzSj2fFUka59Kv984e4m8ZF5IskjfN4FodOqyXEBQ8NBI0STsKLRDydF9nMo+UzL/IDH5di"
        "nsZL4WGazGR6LxnMMslFLpdJiAlOCWxfELrLIBfSmy3UYHcgNJYijwEmjGeg6kPsg11FGuRP4uPpQARRlnsRkAgiQCt/jmWu"
        "mHoehU9iGfvyjZhLoEC0Lr18tsD10cVEpEUoxclxJnrCl9ksDRJaldVhTb3ZHaHASwEpFPM4FednY3DKD2g68FoWuUeX4NDx"
        "yeTo9HwyPhZHJxdHV6eHFyeXvwm7LuU9MZnFiWwPGagid19LTAQZCE9zyDB8asXAZxlkJJZbj4htcICxD+PbYCamElhJIe+9"
        "sFCYiMsFQOH3YfHUMup2AEwBJJVJnObiCOSI09BbehCiX6oTHmdFmGdiMv54eHF4OT79rcMTAIkwbtXWVUgTKVmeFrO8SAl3"
        "MQulF3VEFEfdWZDOitBLxVJ6WZFKUjVCULYqhMHoTIZBJAnYx9P9NwyRpMYSCOM4gean+vEAnD58e3pyOSYEj8Xbq5PT4waX"
        "OyBjLq4P+jeaz6QpvpeClWFwG0m/bj4tzC0ymZU6p40PsDogO4BWTlPp3WWk7V6Rx0tW9iRIGGWmhTbJgqzFcNiogmwWxiy8"
        "aRjk7A58MS2C0O+I83fvTk+gROdnxN0ETAHLoGalVbVKM/flXEYZm6SE8LQfER+LMMQ+Hqzm0ZvBrrxbEUdaUw//AETxtw+O"
        "OJmLp7hokfJIEJbEQZTDzyhVxmIiSDNEoyYfE7ARNrcIbhfiBc90mUFe+AKc+p2eEjHkrhgBLX4DR8F+iNO7ILoVXka2xU9I"
        "GwjwtLhlWUKIbw+PfhmfHU+GsLludxGDwB4uWD/Nz3modI3cSCpsRdvPH69AH3uwyeS9yIsokmGboWD/TM4YWK+CtOk7ZmAW"
        "bIogGhMstS2Li3Qm4XivJoc/j4XNvsz3ssU0hgb12oTtD6RRKZFD0H49fMeOYVtpbXh/cfj3q4uxe3F19vb8/Bdn6fNcT+Ox"
        "TKAlhLqKIUI5oqz3lTDySbspI8xFEeDX9YKemdxNB92a7g0HB9Pu5wcZdaGaeQD1qUAw59nJ8lWDibvGma0/9X8aKE7cBUnp"
        "zGHcAauzHcXMF+/eC0LgIdukMYmJZdrPd2M46U7dWjK4ldniu/lh+PBPsIS/4C7S8WufYzKL0HXnBfyTdF0RLJkEL4JKesrr"
        "t8xYegsXnElzP8vuzeXvWRyZ67SckD1l5hL+uhwu0jAMpk4Dlh5L5ecCxtzSipVLWmdwMvcdhvZHHEk1L/HyBRabaR9x22ph"
        "b4ceOLBwmeZ2v0Nu2KaHNsgNQhDbJhwglnZbAfLIgtyYzcqdhQEeGaDK1o54DJAW3v6r1248Z6BB4sITkehI5lH82RuK8UF/"
        "32DHrtOFR6cEgNyw4Z+KBPqBJC+QLnHzh/TdO/lUxdoG0NbF+fmlGImdlOiP1vkVTeGZPWEBumfRBYUVix46yzs/SG01Oxtd"
        "poUkBwf1d+M7vm2TD5qM3ckYYWQkDvZbx+N3h1enl+7bw8uj9+MJBgevWnzjTk7+Psb9fr+FlOrEfXf44eT0hKdcW9M4Jjpd"
        "2FvkWx1hFRF44FJY41uSpbozU+pphvqxZJrGabVmIR9dk1wBGA3B6czuAAP6kz5ZNy3kc0084KlDeGgFIEP+qK78eFnB9XJI"
        "c1rkGiGAab29OP91Mr5w348Pj8cXBGrF+FlXUKvuIQwvt4bCtj7Ef8BzeL1XTl/Yv4KO+CETZ5di0Hf6bwQGXh+8EY+vD9ri"
        "MElC+auc/hLkvVcv/+y8fC2sbZKxg/3L+8sPpx0RBnfw63J2F7fF0QI6JXuD/QOnT//ExJsjs9WArLZinnU4m8mEELNy+Zj3"
        "Fvky7HjYN1AM6z3SyN7j5ugyfPN51Hf+sfNj70e++slqAOyeetFt4d1Kggx/eTXpyEgtwcQ1dHPgTn6bXI4/gE+2WvlbXMBz"
        "IAoiUJkEDamvlyLbTFVs5XBY5e/wjb2T40kta9MMspRT00EC+jg+m5z8bVzCIx/KUTiIKH4HWRxybhHP55xBwdk6BpQuHWBZ"
        "9/CNtUwdOkAxqqwFCvK95Is526VMwyvx8UJkP/5TN08pYvoI6JRgx6kjzos8ASmUxgDVf56cn4EJqfcEXJAjPJQJlQZEXiS6"
        "zSgOIDgsvfQO+sNZIsVBygu99MkRY6oDzFJOWafE2HkRzXTGreEh9UE1IWk3irzarVKmL2eLKPhM5k6p6z3AErdMhqrLE1oT"
        "40+aOVar3brYf1aqsKogUvLL8IEtIb0uJJeR2JhdBt9NqWlUjXBKIZpEzmyRx6Gvk03K7JFrcvQ09QptA3Q1NJRV90EMHaGS"
        "j/N9I2gqTeB405hrLlNDUZpmSjGdrHvLEprWhUwuPQTvWQZ15axuSUUFmaXSCu8+DggWwrTIkCQGc5QeXIUYrABIplHWgTYR"
        "s/xgPpfkepEUzGLS/16WeDP+NJJo6hBLEXpTAlT4K73RakNa1tnSH6StoRep4oeE2VJu8Tf37cXJ+F3lz5p+elgOdNkVCh6m"
        "ihnq8rsqJ4xvqDt0rLs6Ozk/06t2zt/y+EM19vxOmm0eDCyEDRl9NwDr0WGob7+EwGb4wJpd22Ga50OaiMkInsvgEfCMvAyo"
        "ZtgZlgNdHti9fRWJhrUb6kBUUDk8Dc1V/VkVsPD4+PyDJrQ2YzOKDWtD3YpIrBAPQb4gfybvSRkXcAmhTJUrb6G+EunAVe0J"
        "W5mCmz8lcB1zbxlQcnrHGb+A7UJjI9QD1s86ERWru7XgdBqiEquGzl2r5Tfrysk2A+Dcsle1/ZwiSWRqt9ftL/tV2pJsRDvT"
        "odU2VOwbKvSGHeM9XPYohgr2LiNhvRGW8zsKQrs5SwTz5johQ1i8hVJe6t4BuL3BkI/GIemth5+ilb5cf4o+RZuU110bVc5S"
        "oTUUK/7cueaLLo5FXDo27dFQ7JLA2f1tQmNvlml3ZjzXbm9kaCo5zTm9yxKxMcEwFlNHjUyZH/KzPH0algjcYxqVEg5rRTWJ"
        "fsD8IDONLfue0qIsbw8byGu2X1Om/9jmiPNI4e3+hqfJR0pi1A6kOscSxizH5C4qOImXqUYXpSJIHNVSAhUSKPvRYULstv60"
        "Opa5rO0I1J0MmVVOsS2z2zVEQUjIwTfksp9gWteUfd5YG+TQTBnZYVv8Bcm1WnPdvxGjET67g5tqBECsF58sa7iVSIaCJg+G"
        "mN54Buoc5H4y8u2wXddZPIAwfxD/9Z//gd+t9kBZxNu6rm3rmf/Hv8CQmksNlMh20iLKKp3nICyjW0ow4CrsKUp26r60h5Q3"
        "eMrSRJADmiI/Ewf9l8zXCOaSPd8ZDSKdWWUJamTqPiNWpJluiwHeAz2E/XFDRxxuNF81Tg9xgeRGc36/31f9RDDb7Jpx6glw"
        "SDOCPIMR+aSXXuEH1H+4RUYR0TbURhBT6kMG1FciucLQnrrzVErx/vLyo9D1LFKFLAY8H4xAAaO7LE4SDpynZUgBNpiyBw+f"
        "FC8NywgD5BLkFNibBLeUb8LVpIBGO6huH08XFDY5A9yrOMa5XK2xa5rDGtQbcE5CJoBmXFegd+ceXE7JzMmxyIp07lGxDQmU"
        "TPdUZxJp14L6eIFK252We3F1OnbfH1/ACiAGohb1sp1agf8p2/tk/dX+5K9edf68buPaot6oc2J8merouEa7bAqmHVFGEMpY"
        "IIoRFXjabi3LutCOX+PdEddMf72LfqPcBAWNMo0jyXiZ+Hl8uffxfHLpUGeGIGrV65R8IIfUEe88BJ7SL2G1r9wI1pMjIRB1"
        "X9Lwsdqx8CI4El6z7TaKlBzH3FoR1eveXz+PVvW2jfO5iHNZhdPMm8vRixft9XYli9oDkJr9HedCfdoYRiyWiDX+iDFpN9ZT"
        "fN3GjdoZFUiFD244NZP2yvqMjMeE2Lajh9vfhxiRDeW1euAn7TeiPxWqisO7QDqe77tKL23rKOY4271EFsP9hXqx3X14eOhC"
        "fstuiby/AZOke9dBWIR0NxoRDnzEEoFl+DUkaH0TKnvHDbpxG8Np2LivVFt/tkk10+2NCGOyK+dW5hs81pFWb8IJuUM+gmMt"
        "gdsQqwZFFurLxpOFn3JOZlIyqOTderi6X1sN7tjS0c5AYHQFuWv+NEk3rnxvJErX4KCC9b0wtGmrLRLG/AGBDZu9uLen435/"
        "sIuI7qAe6tXgiMJKk+TKoqnd1lKwj6VfKBWR9VAD17soUIOyd8veEFxOOrUvhAHe0nuQygviocMgKZND6sWpqZ3J3KS0mi8M"
        "j5zK3Cqj2yoNfM1dXBFzAeSG2GqXBFxbZid6h9qjUoK9OLcQ9BbWDeFpyORU+fqmkWqUTpLRMClkGk+l++DNtcM1yvhT6WYb"
        "7uwZPW6uNYrc2kgUS85/UV+HX1u2qSNf0BMNQTlwRTA4yWWOPdPdZW7XI6aVtRYC3FRlAyDJS2Fu8LhSRhudUh2wFOmIf266"
        "rznmcfeOOK1eU/HVvstVX6ZjCv3vq5jiLd0Ao/0ywkxVShvdSlsjspHPguNmD05WFZYbCk9vCssRbEIBBsRdq/3+xFBooH1T"
        "n4VHsNaKg5lko6ma03tiWiXvqPxy+Aho6Ir5t75RrBOr6d5g3Vtp9NdC17CjFT7XCupoRX/XNR+8FThVJaME5ZiXLrYWWL1Y"
        "NqLriKpD3u6Isj/KAvS3vM1FEZGuPucmK/KEWMn1RrSgN3dBVFT+k0RHBw9Iersrs++V4bYcGa0BM0V6kU3bNbGK70jrPNRc"
        "mGTeddjJQCt4exMPqoniu11hTWmuqVoqEJrRxNCOGJjd2u0dAWuDQcykAMknFkVkE7oDQKAHHVP7VSatayJd+QzrxVG3rELK"
        "vPZ/oMjZ4I0yai4Sasa+Q0TGsRqCNnPZhqtIBtusqjz3cNe7iW3L2LCS/S+YSdWNGWgMyTL2n7GMzZ9kv1S2zY7Cfvv5ZfHd"
        "fke4DRXc36mCG1zAuucJ3dKfmk8l8Pv1oT0x2Ano20z/OTegXuXCAlDXAIM7KRMq1nh4MNzhJejN4mjjVaPNRGyZIk2llwMI"
        "Nd9gj5oR2ybJrLHKxMb6VsukfSmTtYFHc4nxU9+697/p5FEpW7uRhHwxNppS0MNiE0sTsM+8CncO01s+IfaR7lK7VueNXBc1"
        "tut2vqg9xk8vuc/mzkIvy0Yl9Avv4biC+F6GyTszta2R4VTf01jYln7Zb+3eFbR4RZiPrO8/E2E9uyEdgKAqycAuT0U8v4Ra"
        "pVhC0hoFlPKYxYPBwcuDL5Fmzl5854a10xk796UTG7tZtgDTUe/VjrJsnYqxyz53+fLs4+k+v77kRQgFgN9vO8/jV57JAHYe"
        "d4lGqvXv5sg1rS+hNqEzJrRnvVPXOGrzpjw99ZWjJV9AUOdNO5m3cTDgWRgqoegmMu3mqiDehvWy3392vVfki24e38moJv0z"
        "lDp6RXpLfhcLTa5zi/qvxc9082vUOMdhk86QsWUOXSEQQj3UPV116ETbwuUN1Wh1bzzIrApzYM/szmXrs3k2X+r9a/GW2yqL"
        "PE+Gvd7KTKSHhAMq2/oQoaFbKjr0j9RRMZoTxa4+x9PIDhqJeVVM1TBoP5dUXp//crPV8E3p8Cm3F71crGpg1jsOiCHtqoWb"
        "7fZNba9fDy/OboQ5CrUB+g18OEoxBDJzzrSuus5GSCt5o8sqTlD6uKeA6tAf3ZrgSIa/Ob0huKHCGBWxemQw+xTpV1i0OR8Q"
        "tpXeCiUZdeNCibloW7eF4xh8MpeS3QwkUEtMobRV3VW6QedQPocBlLlxauYbAgbDKEvCXXjpAnFnTdigt0YtVfHfQe0jU/uo"
        "qH38FmofM3Ig9ZM5//u0NupfkjiwRipGyLfqaQw/Ain8DJ/ab6gDJ/AbV+oklTlet4hDasQ7s+xe2Sc31bj1oNaA3AcQG8kH"
        "egM0snBt3h2PrCKfd3+yuK82rwyES8vs3uH+f2rPKyV/0GPxg31t3v3r80rmJWntVlW2fMxJqnNN7CeeiSJW45UmzQ69KWUQ"
        "wiqP2tONOu+ae7d0Yw61qYNw1k27UXMmz2Ziqh4Jqqyr6R7qdH4FCBdoVi17sZ7XpboSlkf37KR9o9UDOrBDxEozsl0ixqP/"
        "j/Kl0wSEs6sq4LpYqh0y26i8oX4fMS33QrYAWPEeG7N+/RAFczrDY06JsMJUajF8TgI1bRnW2b85wQ9uAR5zEEqpjWxbegS0"
        "mMNs6kxDlZq6fGaM9lZZbNcLLFapchvqVdZXbDx2wviBzhPUwFOUQSHBMRiQ59aqTAtMSFaxuLZGBV9XRhQcCR/lcrZnmOhI"
        "cRjT6rlAVdNzf5SSme31pqGQufz+gPcyQqvNphoqjUPg4lbH9DF1ZdFXS+i+SOngXt/p07HLOHHvcDfQ1wldO/11gylAjVVs"
        "WLXbas833LCR9KZ3rglRGz0jxaFvyG0n8sltHR/UyKMeYRr1WNmcquNodNmtzywVfGtvOjvgkk4QC4m7drPt1YPKP26M7W3A"
        "FAP8P6jDlqGX0GkmiDWO/KyEXUs/RBcZCS/dsiMsLPIZFpnzzcjsHmxzxNnBs7YTZLEqEhtaWybwrvkmCZI2Po1qErPygM0D"
        "fUWk/AKN2P4CjfX1SCysrdyw/KbNm/ppTf6qB0ogR1yob9R8E/Cvfe0mk8BfvZiuf6GFyqoaT5Ae8+nU9+Y0KSeM4kzSdyT4"
        "LCd/HYEObMJVd+gLEvzJ31HAVhTYsVWoNnC0wa83AsBWrCgi1zhLh86XWDo+fC0m8FkUv1gmtlmOeNeBv/KRPo1M/qKSNeSm"
        "Fr2THFniR/H6oJnHHZqAKlZNY0FKx2nsqrS0NVCnXG9VWtraZO0G2uY7+kxo10PnkLTrWW+u0RaiMSgNZm3yL/5GmjocZoi9"
        "frFhly9ufhz0+0NnMF//aQP+kcrFBOCrFKuJQMWWVgtu1XXpAIfr8jtu16UOjuvqF92qndP6b6+WAfyqOAAA"
    ,
    "azure_ollama_client.py":
        "H4sIAJ4HjWoC/9VY23LjxhF9x1d0sA8EbAoSdXHFdDFlrUx7VbsrqUjuJo6iokfEUMQKN2MGkmiFrnxEvjBfktMzAAle5K0k"
        "L7FWK4KYmZ6+nO4+M6/+sF+qYv82Svdl+kD5XM+y9MhxXdcRv5SFHGdxLBIxnsSRTHWQzx3n8qJPaiYKGZJ9S9OsIC3i+yi9"
        "I52RSOnSrCIliwdZtOlxJvVMFhRpKspUUZY6cTYR8SxTmrA4S3nRKW9IP1x9oI/vA+pj5ZzuZCoLoSPMUJMiyjVFKelZpEiV"
        "kZaOd3gyrubI8SQLZaVuVuSlgrptOvxqNSGUMldS3q/GfYqSPCu0cqZFlhCUlKQy7CBJFyJVPNamQuoCokQKk7NUF3CKDPea"
        "ukmtYb2iOHqQDlSUT2Ki4zksk5THYiIDxzmbiVzDC0dtGsqJWXcUnED4z2VUSNUlLZOcJcINvYM2XJmP73udNoXZpEzgaHgc"
        "2oeq7bAmOdbAv1aBbGp0NttSwn5ARO72w+hOKh3QiD1WRUumiNdEKl7gTKMCIdCzQkpjXiHhmtCMIaxloWdwOGIqTTQQPEpE"
        "Gk1ZqOP8+c2PNHpzPqT+X86HoyEN+1eng9NR/92P9P3g8j11OivXywehoOjK812HeMIX2PKxQCjtnodHNI2eYGj/4+nw/PJi"
        "fDoanZ69HZIn6EEUkYABlcys8NkwCTlYqWDgTMbhXlZqspuQ13CzTym7jl5/ePe2TVORRPF8T+minLC7QwhphBPe/PXk4OBL"
        "/CfvDN6kdwwr37jo1yOMHPHId4DTEHDy6eLyY/8dZORiHmciVKyYAFYVCQqj6RSwguKfsts2oytiG7TCJ1LhsUa2MtIZ2xBU"
        "JditmNzLNISvR4PTi+HV5WDUNX6qEuzNaHRFp1fnvFMUYo8IWcXptEovFtrIL+fje9rbwxuAkwXxnC6jnCYzkQItAV1lCDkm"
        "2eXaTPv4vgX9whCYg18zoIGEkxcR4IfBC6k5O2g4fEO6TFMZtzmtBatxj0wJ2cqL4Q/WgWkGI1EqZKykTWfp5FEu4wjJUmvh"
        "wAmvrBl4qqDbq+w+M189VrDnLk1128SG9Dqd46Nj36y3JQUmF1JMZnCo1bypKXmM+QT5FWIcrkGK0elfPwz648GHi9eXl2+D"
        "JPT/Ox2Idai2SUTOEV+atEO5EGXAVg2jJTX9Wzv/M4p0DgL+d7zhC2fYP/swOB/9CKSO+l3kAu8gn/JM1VhCfA0MtK1+eXkb"
        "RxNOflmkEiXkXNNMKATPESUmGKhxspgc5Mpwm2X3VCqpNvxblVMjPFIWeby9Y8wWt7Hk+pOVdzNe2BAOj7AYSOTKEZimZKr0"
        "eDwtOWvH46p6YyFAZdRRjlO9g7azOLqtv35S6DrVs44SWT+XRYxZgSyKrNh4x5UZlc5uGgotJrFQbGA1bfkKBSVC7XEc59vl"
        "O8f8XYsRlzyyGUcoPYhgAzhmjMV22ekYM6EzbxXQkcjlmpnWuZ2OZpQCMWO2B3VvufLEjFbqb45+jdK1/HmFyZQgCScyijkp"
        "uZerOHukztFe5/i17SXKyEvE05g7YcS9ysqy+nGVyqbTsUIipSEGpyiCPHwSHDh2l1uJegaWkKGcWQhEU5pnJcG3aV1i6DFC"
        "vxFQHPAAMPMie+IqJQz6lcHGN5VANNVoyTOiuzQrOC7A6VsJklBhzmZYjc0J4BWLOacWBJkqCZAVgZHIr8ZGPevnv9MF9+6e"
        "+TAzYLvK4gcZjm1XreNhQu+FcirKGBmIzIvSSPe+FyhvvlOp+69//gO/9GHwTlXPv99fY9K3CA7Iip6bb7AeKFDSUzKe+rT3"
        "J/aNhbtFItI1pan7zOOBxfOiu79vvzP+F137zBmwcJ2l1PFMihBoWEkOo4leiZ4hAs/uGUCEBNsbzXPpdskVeR5X9WmfE99d"
        "LBcAd2ajRsAdavzMrt1TDGVF9ItZ795wjN3XFsDPG2sX7qaRs42QQ/8YaNuvWJkBUWlYxv9NLNnPCMnkfmx0NK5uW30NyHe4"
        "HbUYPgd7TEyiWaLP5b3MKxojK4PxLi+ZLwc0MB6yxLMWlAg9MWygdg/zT/KidBKXIb9nkmQTzrcHDdttVjyU20ItDTR9PZqo"
        "gQjfej0PBvbTq/HIyF3sizzaB2dWbntNwm/+VODsGTFLqPr+mgRT1TZUwFekTwru8XObqgJtpWzUdFAmRUV3SyXuM7CMwR0Y"
        "yukVEC7C5ubyacLFsG8+GHEQJTf8IyIU2kGZ8nZ97oHe1lZT97o/GFwObujM9FnLV+rai+K8y4vB31J3hySi86k9v0Vqdeb7"
        "+L7NvWwJpwZ7QMQN/QFBABPaJXIFulbWOHa2eAcAJWUQZWnVYl5U6zupRRSjdz3LxfqMlT9tL4TX2fkBKLzn2leo+dc3Tb/z"
        "KaxH14lBbMIkt1qL6pPYlalIpOtTr2eHbpar44xJ2Yur+SlQKG+Q0XX964MbM6EhtI3s9G/WMwzyPKsWRJod/GZF5Kiaef8L"
        "Ot6bDG49GxUXLSMUZ0h7/mhg5MUYnD4gBEwIEYVra1GLLWr5m564Wbwk47KONMe+S1RBgksQVZq9FF7UHrjJuAFeXb62ub3R"
        "/DER063T7Rvr9uUqcPeUKwzR9eXbm6b5tOS9wdJltazKXDhv07pptU3veac+193O1zcc21aZ3qc4brUWrr/ZmrDLRnNqnHl/"
        "nzSEW1d9x7DVt9rMIJNcV1/UXGmZ1F+kDA2L3WYraCiXfBJd3vQ03BS8cElDeNe81QmW0gaWMdvrJHNww3HqMSvuQQlOnp7I"
        "HDyUbRIVkf7GJp65J1vVlJkolb1ssuQWB4cYmxpWm93ZGxvkD9StbiEgxdy6gUCv5CjMSPmMKSamOWA1DNLmpmqtld5m4bxu"
        "MGGZ5Mp7XoOkLX0gW+ZzvWe61u8YtA8bozYQGLUPm6MaCcKjhkFvDGamkSmMPm+lv8sxZan42O7hbiNwmIRTcttkQiNo3d0x"
        "3yGKgw4hnTY1j1Jrou4KqDEHQPlWcreInEVAj+3RtEzGeSGZc2HOSedwfc5i9XXhBzLlLTy31NO9P7rVkcN0EoHTH/DVbVCA"
        "7XONqS58W6PZQ+aCteA7GK/TtrWvce6jL6njr/eILcr1Wdq1k15tc7E6rVFYud/2GJC7qdkLLAylQILIhz336nI4cv2ttf8Z"
        "Mds4Tr9EzKz5CmfBz9GzrerMi2xLwXMOmFe9PEBKRLm3vrIid17z/iLg+0DTp9tr9xoBzp329U51R9aiauGZJaAAi3nh72CN"
        "TXDBym14MyAxAFuAqsKTMINfwRyG3bYLXtExSqFX2XR8+DV2BS/MyoLMsZrv5cIsbWl7FR9sCQCHiVSUKi3SieT9drvFr+7w"
        "oR2zE0Ddw2ZtOjk44D+H/OeI/xz7L0R2mxGtCFDFiJ9Z/sJSFtPfK95hueUOGIqIsVLBf+0qhb6o03JrVYNgGJ/QczVzUZ2q"
        "G2m7AAHROBp70g/GY+YY4/HCfeGYg0xsUetLYM5jQ3z2rfGYua9ttRbfGI2ZVj/zQzc4mC7UDrvYRYGKpcw9ntegI5+hlSuP"
        "/rDiJ1VvE1O9PIY3Laz9pDaY09QVd4KBsUbBvLXA+IhMDedFXUC3Lzgs0/rtK46d5MxxHF6vZuLw5KtxNvW0fNKrk/VSTn2D"
        "YC8uAzvfTN6s8X4wk0+VPn4tnovEeIqZEtVGPO7YAP19IJPsQdJPP/1kg2rnm7SIUZ84qjGqfynuJOFcDn+qLJH1AUQm0erA"
        "jT2YBovHtfoEtNhXotCKK6znYjO3kU98z8+nqOvY5EhsO86jPdSYQc+vDyXxtqCbBo5YARdHgeBTFqWeWbteLevaKh6dfwOx"
        "4UCx6xwAAA=="
    ,
    "payload_validation.py":
        "H4sIAJ4HjWoC/51ZYXPbNhL97hn/Bxx9PUuJJNtJ72bOl6Tjpk6aaWrf2e5MZyyPApGQhZokWAC0rIT57/d2AYqSEyeZ5oNF"
        "UcBi8Xb37QOy87e92tm9qS73VHkrqqWfm/Lp9laSJNtblVzmRmaTW5nrTHptylG13N7a3jqfS6syIdNUVV6WqRKp1V5ZLcXM"
        "WPH27a/Da1UqKz1GzVWeDU3tRTTnBkJWVa7x0xTWpsbPhZ8r8dJkSrzNZSGFLDPxk1LVuVI3Iloy1o1o7ZdzWWEp8XQgzlVK"
        "Tomno38K7YS6g9VU+0ORvF6tjoWr2jux0HkuSuPFVAlvFf8m3fYW701I72V644SsvSmw01Tm+fI/Qsl0LqxKjc1EUTvagnPs"
        "rPN1thQdMOTy9laaK1nq8noFxyjhvSQXmKLuZOpFh8tM58rBupe6JGdmtKvO4vYWzcxUVtOuwiK8CWxgZs17xWuK2vFG2Cl4"
        "PzdYGYNvsUsFh+F6VbtRQshdzAFSYbI6hzNlCrhpmnFr0dOlMCUZymBlinFVLlM1Ese3yi7h6x8RcQ3kcnN9jbUXGvGT+E06"
        "/OAMezLTpcxjfqhsL0zE4NTUJYLRs+rPWlvOABqPjEpNBVSD+b5IZUn7tKoyluZ5i63NagRlFHYCn+eKI2YVQMr1lFHNl1ii"
        "dMreBgwcJqa+tnAGcGOPTgyHtCKPm2lbCLnKc8ohSTuuncbeB0iXcuitvtWYfv6/t3rv9/NzShVVVF6YGe+UdgKgKZ6y0Ozf"
        "BZnPjDg5vUDQVVp7IFqispAZI3Fi+AFlYWk2MkzRsjm7u4SlYhTLD1EuxGQyq7EBNZkIXRAaMIU05nxwhEV8a9XqsS41BRdp"
        "JGnAr29OJm+PT8Rz8T2+HP0evzzZ398Xa/92Vj6K3JTXyEWCIy8MskjmC7kMOYb8UbmwspjmGEr2d8R5h7LT16Ukf52oYCLU"
        "lfDLCkl01DJAqCUgW2GAQM3gG9KOTBF9aGRIptKcOSZMBaRIU4r1GufEiA64rMm3THkkkLG7jk2p6AgifoM5yK5KUpJgLVHV"
        "tkLmA+oJIvtmcv7m9ck5ULnc3iIwrBqlpqhQoT2bjKc9p3JYboCsKRtNCYYvFRBWDeBQ9GFN1VC0m+h6fzxNBmToTX8QjO6I"
        "C+IX1M1S7ImpMcQWorJmih2+NLeUnUcnP1E1LxQKHZ+nZ4fiHb07eH7wLnBMsBRfPnnHoNDmUTKmJOJqLU8oQhkw13bAVAFi"
        "Szm+pkSdnJ61tlbV6ecIx2KO2TGZd52Ykn1EpSB33pVm4v7M9WQV5nejzwNmbIMlgcHYPboc746Tqx/wNM4e4+/z8LRCh3ww"
        "do9cxDYfCECOdtAspPbwpamuJ+HFFDw2L6S9+RTt+xbUnbcgYBBsrWLs7oq8AQ0AeP+V+b3hsNlp9saP+gn5+7V/O2CXolAl"
        "0tLcKKrTTyxeEiYAokPqS+vv313uD/8th7OrD08GH9eQe2j9uboDpaA8ZL69dYUsB3V9McmfUaDk8P0V/cFaV4++uMqO8BKJ"
        "VKnys2ibkm1xtL/BW3VLWM2BQ67sZwz+IW+lQ5Oq/OFXoixzKk3UFCi6iRTfKAS9ITZNoVuIGvuUgb0vBvy7p2nz3VPV/GPn"
        "7odx1v/KJnZiQ83EM/ECxY3taL/8vIuZSWvKjWaB8jSL/ng0Xqy5chU49Yz7KVXrHlodJMLQzIbo1vX1nCnzRl6DNbpRzMvo"
        "rNQbpnnsquD0acgCKu9WUQ3PDsS01nnWF6rQTJ3aioWxN2RIEvs7nQVWiYw9IlunU2qspKnKwDjIce1DWyeVcSieHHxHfbHt"
        "+52Gs2ZBpAYqQWicGpC5XE7BckG9RP01APtoSK5MVSBup1xssVEFBH4n/cE0PlUpWjVvE7XsxdGPp79dgOnaroP+xXojLiEK"
        "7Vgs8a/E/Bc/vzn5ZXJx9Bo1sVkMez9QJ7xBmrxoQ0NhmZwdH52fnrw5uTcjBjrpJe0T4oy2IArV4GPXNXo3zxvNAg4f4Gr6"
        "a+o8w6ebx4dSEQmbZmVF7xbi2lBU8FYLufaN5APssorQrkGeOz9oSuAwaABaOeis5Oiv+XLQsCjjBzRZqNGiyhU/o6FxRwUR"
        "QZygbaMZd9PxTnFraG4QTpKIBRK3AaRLU7NSc0o13ghXF6Di/mpis47GgjtUQx+8WzzEfbdWEMrwTG/5myfB2osp2LQew7k+"
        "0WVrvL8eIYgRlBaKUQ0dDgoqO0QtZo68xtkEAhKS0FRRbDvwDokWpD6dSoAEGIPkuCNVCBInKaha3YCfIUa4wyLH6bAhS9lm"
        "I6EGjKCMy6iRWVocn1wcn7w8npz/fPTf4/tZxhT54eng49g9/uzz6BG4wT36++YGz9SsdhK1vof08nIIdvCel2/VWSGXobIp"
        "sAp6i2pyrZSRxq9+Oz96+0ASa9FLWWQ2+NgFTxn6i+SrSzoT9BtKS2esXVISUL3pJkxAnTnt/FoCB6lHkgiHnIaH0LEC2YJj"
        "YdXIa0IbdbLciOL2VqZmgk9TPSrtQxLyfTF8QZ+HwTgk8oXVxUCQ8kLuWYtaz6g6/qyRsSwLbYHjFCJIi1Me4FcHBf7ql5dE"
        "DtDnwVR7HOViCueDSGnB/XBGI30s5qYw1/mymgszRRjiuWwFfbAHjsvqVGWDlqIAAKlRLl3SWFgcBJfCNTD1gt6sy1fKO13g"
        "0JZEHD0CRTCMHGnmXn/t7QjkSEe0XjK24xIYJvjb3xyoZ2gYALIvXkD2sxT0l/tX4jnmXw4Prro32GQCXfIuiRivlrk8OMTA"
        "TbNApbbYexeweHhVbcly2AaRcCek5LtAehxr1SVJ1QG9u+qiehbs9touMohny/6oPWQCvt1d4EZH4NUgdD+Fd0scH8RCriRX"
        "lLaIajyvouSnCEWBzXoT+wsNcnwg5wNIiD86R2w9XSQAJeVwu78OpQjGK9QlDo4JnRCX3Zy24EZOQYDMW3z6DxvgbJrYUOrR"
        "UIlIUCDb2Z1LUB3xlPegxVniDcT7HNvpfSg/9tc2BMUSj4VfmU3KYH0y5AUzpjU5sZAd8sECAKKLowIr5GV4RXmD3EfcmAFa"
        "/D6HaZtjD0PTGZ+Q8ZUvQ8TYfoNqooEP/WttvaTLBSTLj8evTs+Ow41Pe+Thi5XDIGRCOQfC6faFhoIW1trilrogGnLIOzdb"
        "3jNH1TdVnWiKnNqqlDaFVmLl25NohcaE1cwEkj2ixSCjOyKj7uG+WrAdMsoNhFuvT+TQS56xJRJFEEnx+S95EKoiqqm/siXG"
        "fxX87m5BM3yrNhxzbxAzz0V1QNKZr6AiIfuYnK09t6kioIhSRaPUXQqRRrGeqUUQDjFG/Aw8V7g5NBV/j4N5EPPwvzju9xRC"
        "i0Nr45uAaF2dsKt808chRoKRP+v3G/BijYypASR0nE8CeXZnxI3CRNvr2bv7IWKitHeUFbxS/wvcUZrJh7V1P3bXB8nmHQSa"
        "6q02tQvFdSjo8gn0IfNqjsOC12lEub3J4ftDpgLH95CttdK0Z4CqLlNfy3hpCSFwQy0XCbKm3wabrBTjCd3O94XPSdf20pF2"
        "7EYv7DyljW8SsavQ0UmXhRkHn4xrAQ22SozpcxK073gjvf56xgQXXojv93lkt8Rzsf9wbjB6E7qrCeMnRM4u2WjbF7amoUnX"
        "vaNceq+yyY1abvTwT6XXT3QvLTDuMMipx5DwTs0MlDseTZ7LittxS9UjcbEwq/8B4KILtjI9m/GJA2FBMMkKdRCaxBfp3Ibj"
        "/baThWqv4wlcvhyPZ8YImuP6S30rx/nSfcg3X2F4dxOGnFn7D4RdOiOgjvym7Fq7Sx2tAOoltGdwYJsAo3bz69oMatbVUzrz"
        "O7rrSgT++P5DGur/I7XhPYkZAAA="
    ,
}

def materialize_scripts(target_dir, verbose=True):
    """Write the embedded files to `target_dir`; report what changed."""
    target = Path(target_dir); target.mkdir(parents=True, exist_ok=True)
    new, same, upd = [], [], []
    for name, blob in sorted(_EMBEDDED.items()):
        data = gzip.decompress(base64.b64decode(blob))
        dest = target / name
        if dest.exists():
            if dest.read_bytes() == data:
                same.append(name); continue
            upd.append(name)
        else:
            new.append(name)
        dest.write_bytes(data)
    if verbose:
        print("Embedded code -> %s" % target)
        print("   new %d | updated %d | already ok %d" % (len(new), len(upd), len(same)))
        for u in upd:
            print("      overwrote (differed): %s" % u)
    return {"new": new, "updated": upd, "identical": same}

In [ ]:
# ==========================================================================
# STEP 2 - SETUP
# ==========================================================================
REPO_URL     = "https://github.com/Judge09/AI-GIS_dashboard.git"
CODELLAMA_TAG = "codellama:13b"
DEEPSEEK_TAG  = "huihui_ai/deepseek-r1-abliterated:14b-qwen-distill"
OLLAMA_HOST, OLLAMA_PORT = "localhost", 11434

def find_root():
    for base in [Path.cwd()] + list(Path.cwd().parents):
        for a in ("data/eval", "data", "scripts"):
            if (base / "dashboard" / a).exists(): return base / "dashboard"
            if (base / a).exists(): return base
    return None

with stage("Set up environment",
           "Locate dashboard/, install Ollama (Colab), and unpack the embedded "
           "generators + validator.") as rec:
    ROOT = find_root()
    if ROOT is None and IS_COLAB:
        sh(["git", "clone", "--depth", "1", REPO_URL, "/content/AI-GIS_dashboard"])
        ROOT = Path("/content/AI-GIS_dashboard/dashboard")
    if ROOT is None:
        ROOT = Path.cwd() / "dashboard"
        note("No project found - creating a fresh %s to hold outputs." % ROOT)
    ROOT = ROOT.resolve()
    SCRIPTS, DATA = ROOT / "scripts", ROOT / "data"
    EVAL, REPORTS = DATA / "eval", ROOT / "reports"
    for d in (SCRIPTS, EVAL, REPORTS): d.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)
    note("ROOT = %s" % ROOT)
    RUN_LOG_PATH = REPORTS / ("run_log_%s.jsonl" % RUN_ID)

    materialize_scripts(SCRIPTS)
    for s in ("25_generate_codellama_corpus.py", "26_generate_deepseek_corpus.py",
              "payload_validation.py"):
        check((SCRIPTS / s).exists(), "unpacked %s" % s)

    # GPU
    GPU = False
    try:
        import tensorflow as tf; GPU = len(tf.config.list_physical_devices("GPU")) > 0
    except Exception:
        rc, out = sh("nvidia-smi --query-gpu=name --format=csv,noheader", echo=False)
        GPU = rc == 0 and bool(out)
    check(GPU, "GPU available (generation is very slow on CPU)")

    # Ollama - install on Colab, then wait for the daemon
    import urllib.request
    def ollama_up():
        try:
            urllib.request.urlopen("http://%s:%d/api/tags" % (OLLAMA_HOST, OLLAMA_PORT), timeout=3)
            return True
        except Exception:
            return False

    if not ollama_up():
        if IS_COLAB:
            note("Installing Ollama (zstd first - the blobs are zstd-compressed).")
            sh("apt-get -qq update && apt-get -qq install -y zstd")
            sh("curl -fsSL https://ollama.com/install.sh | sh")
            subprocess.Popen("ollama serve > /content/ollama.log 2>&1", shell=True)
            for _ in range(30):
                if ollama_up(): break
                time.sleep(2)
        else:
            note("Ollama not reachable. Install it and run `ollama serve` first: "
                 "https://ollama.com")
    check(ollama_up(), "Ollama daemon reachable at %s:%d" % (OLLAMA_HOST, OLLAMA_PORT))
    OLLAMA_READY = ollama_up()

---
## Step 3 — Pull the models ⟨~9 GB⟩

One-time download. Skips anything already pulled.

In [ ]:
# ==========================================================================
# STEP 3 - PULL MODELS
# ==========================================================================
with stage("Pull the models",
           "Download Code Llama and the abliterated DeepSeek build via Ollama.",
           params={"codellama": CODELLAMA_TAG, "deepseek": DEEPSEEK_TAG}, optional=True) as rec:
    if not OLLAMA_READY:
        skip("Ollama not running", produced_by="Step 2")
    else:
        for tag in (CODELLAMA_TAG, DEEPSEEK_TAG):
            rc, _ = sh(["ollama", "pull", tag], timeout=3600)
            check(rc == 0, "pulled %s" % tag)
        sh(["ollama", "list"])

---
## Step 4 — Sizes, and the payload plan

Change `SMOKE_TEST` to `False` for the real run.

In [ ]:
# ==========================================================================
# STEP 4 - SIZES + PLAN
# ==========================================================================
import math

SMOKE_TEST = True      # True = tiny/fast plumbing test; False = full 1,600-payload run

BATCH_SIZE = 20
SAFETY_x   = 1.5
CL_ACCEPT_OBS, DS_ACCEPT_OBS = 0.784, 0.101

if SMOKE_TEST:
    CL_TARGET, DS_TARGET = 20, 20
    CL_BATCHES, DS_BATCHES = 3, 6
else:
    CL_TARGET, DS_TARGET = 500, 300
    CL_BATCHES = math.ceil(CL_TARGET / CL_ACCEPT_OBS * SAFETY_x / BATCH_SIZE)
    DS_BATCHES = math.ceil(DS_TARGET / DS_ACCEPT_OBS * SAFETY_x / BATCH_SIZE)

with stage("Payload plan",
           "State exactly what this run will keep and how much it will request.",
           params={"SMOKE_TEST": SMOKE_TEST}) as rec:
    print("   %-12s keep %d/type  ask up to %d x %d = %d raw  (%.0f%% typical)"
          % ("Code Llama", CL_TARGET, CL_BATCHES, BATCH_SIZE, CL_BATCHES*BATCH_SIZE,
             CL_ACCEPT_OBS*100))
    print("   %-12s keep %d/type  ask up to %d x %d = %d raw  (%.0f%% typical)"
          % ("DeepSeek", DS_TARGET, DS_BATCHES, BATCH_SIZE, DS_BATCHES*BATCH_SIZE,
             DS_ACCEPT_OBS*100))
    print("   %s" % ("-" * 60))
    print("   TOTAL KEPT if targets met: %d payloads" % ((CL_TARGET + DS_TARGET) * 2))
    print("   Duplicates are deduped on a normalised key; near-copies count once.")
    print("   The generator stops the moment a target is reached.")
    if SMOKE_TEST:
        note("SMOKE_TEST is on: this is a plumbing check (~%d payloads), NOT the "
             "reportable corpus. Set SMOKE_TEST = False for the real run." % ((CL_TARGET+DS_TARGET)*2))
    rec["plan"] = {"cl_target": CL_TARGET, "ds_target": DS_TARGET,
                   "cl_batches": CL_BATCHES, "ds_batches": DS_BATCHES}

---
## Step 5 — Generate

Runs both generators as subprocesses. DeepSeek Round-2 (ModSecurity feedback) needs Docker and is
skipped on Colab; Round-1 payloads are still valid.

In [ ]:
# ==========================================================================
# STEP 5 - GENERATE
# ==========================================================================
GEN_FILES = {
    "codellama": {"holdout": EVAL/"codellama_holdout.csv",
                  "rejects": EVAL/"codellama_rejects.csv",
                  "manifest": EVAL/"codellama_run_manifest.json"},
    "deepseek":  {"holdout": EVAL/"deepseek_holdout.csv",
                  "rejects": EVAL/"deepseek_rejects.csv",
                  "manifest": EVAL/"deepseek_run_manifest.json"},
}

with stage("Generate held-out attacks",
           "Run both LLMs to produce the frozen evasion test corpus (test-only).",
           outputs={"%s %s" % (m,k): v for m,d in GEN_FILES.items() for k,v in d.items()},
           params={"cl": "%d/%d" % (CL_BATCHES, CL_TARGET),
                   "ds": "%d/%d" % (DS_BATCHES, DS_TARGET)}, optional=True) as rec:
    if not OLLAMA_READY:
        skip("Ollama not running", produced_by="Step 2")
    else:
        rc, _ = sh([sys.executable, str(SCRIPTS/"25_generate_codellama_corpus.py"),
                    "--model", CODELLAMA_TAG, "--host", OLLAMA_HOST, "--port", str(OLLAMA_PORT),
                    "--batches", str(CL_BATCHES), "--target-per-type", str(CL_TARGET)],
                   cwd=ROOT, timeout=21600)
        check(rc == 0, "Code Llama generator exited 0")

        ds_cmd = [sys.executable, str(SCRIPTS/"26_generate_deepseek_corpus.py"),
                  "--model", DEEPSEEK_TAG, "--host", OLLAMA_HOST, "--port", str(OLLAMA_PORT),
                  "--batches", str(DS_BATCHES), "--target-per-type", str(DS_TARGET)]
        if IS_COLAB:
            ds_cmd.append("--no-round2")
            note("--no-round2: ModSecurity feedback needs Docker (absent on Colab). Round-1 only.")
        rc, _ = sh(ds_cmd, cwd=ROOT, timeout=43200)
        check(rc == 0, "DeepSeek generator exited 0")

---
## Step 6 — Audit + checkpoint

Reports counts, acceptance rates, and re-validates every stored row against the current validator
(so a corpus made before a validator fix flags itself). Then prints the copy-paste checkpoint.

In [ ]:
# ==========================================================================
# STEP 6 - AUDIT + CHECKPOINT
# ==========================================================================
import csv as _csv, collections, importlib
sys.path.insert(0, str(SCRIPTS))
import payload_validation; importlib.reload(payload_validation)

with stage("Audit the generated corpus",
           "Counts, acceptance rates, uniqueness, and a re-validation pass.") as rec:
    STATS = {}
    for model, files in GEN_FILES.items():
        hp, rp, mf = files["holdout"], files["rejects"], files["manifest"]
        if not hp.exists():
            check(False, "%s holdout written" % model); continue
        rows = list(_csv.DictReader(open(hp, encoding="utf-8")))
        n_rej = sum(1 for _ in _csv.DictReader(open(rp, encoding="utf-8"))) if rp.exists() else 0
        by_type = collections.Counter(r.get("attack_type") for r in rows)
        uniq = len({r.get("payload") for r in rows})
        rate = 100.0 * len(rows) / max(len(rows) + n_rej, 1)
        print("   %-11s accepted=%d rejected=%d rate=%.1f%% unique=%d  by_type=%s"
              % (model, len(rows), n_rej, rate, uniq, dict(by_type)))
        check(len(rows) > 0, "%s produced payloads" % model)
        check(uniq == len(rows), "%s payloads all unique" % model)

        stale = collections.Counter()
        for r in rows:
            ok, why = payload_validation.validate(str(r.get("payload","")), r.get("attack_type",""))
            if not ok: stale[why] += 1
        if stale:
            print("      re-validation: %d/%d fail current validator: %s"
                  % (sum(stale.values()), len(rows), dict(stale)))
        check(not stale, "%s: all stored rows pass the current validator (%d stale)"
              % (model, sum(stale.values())))

        man = json.loads(mf.read_text(encoding="utf-8")) if mf.exists() else {}
        STATS[model] = {"accepted": len(rows), "rejected": n_rej, "rate": round(rate,1),
                        "unique": uniq, "by_type": dict(by_type),
                        "target_met": man.get("target_met"), "stale": dict(stale)}
    rec["stats"] = STATS

checkpoint("NOTEBOOK 2 - LLM HELD-OUT ATTACKS", {
    "RUN_ID":     RUN_ID,
    "smoke_test": SMOKE_TEST,
    "codellama":  ("%(accepted)d kept, %(rejected)d rej, %(rate)s%%, met=%(target_met)s, stale=%(stale)s"
                   % STATS["codellama"]) if "codellama" in STATS else "MISSING",
    "deepseek":   ("%(accepted)d kept, %(rejected)d rej, %(rate)s%%, met=%(target_met)s, stale=%(stale)s"
                   % STATS["deepseek"]) if "deepseek" in STATS else "MISSING",
    "total kept": sum(s["accepted"] for s in STATS.values()),
})
print()
print("NEXT: notebook 3 (end_to_end_pipeline) assembles these into the benchmark")
print("      and evaluates the detector against them.")
if SMOKE_TEST:
    print()
    print("!! SMOKE_TEST was on - this is NOT the reportable corpus.")
    print("   Set SMOKE_TEST = False and re-run for thesis figures.")

---
## Step 7 - Export the generated files (always run this)

Copies every generated file to one findable place and prints its absolute path, so a run is never
lost to an unexpected working directory. On Colab it also **auto-downloads** a zip to your browser.

Output lands in:
- **local:** `<repo parent>/generated_output/` (printed as an absolute path)
- **Colab:** downloads `llm_generated_<RUN_ID>.zip` to your computer

In [ ]:
# ==========================================================================
# STEP 7 - EXPORT (auto-download on Colab)
# ==========================================================================
import shutil, zipfile

with stage("Export generated files",
           "Copy every generated CSV/JSON to one findable place and, on Colab, "
           "download it - so no run is ever lost to a stray working directory.") as rec:

    print("   Generator output directory (absolute):")
    print("      %s" % EVAL.resolve())
    pats = ["codellama_holdout.csv", "codellama_rejects.csv", "codellama_run_manifest.json",
            "deepseek_holdout.csv", "deepseek_rejects.csv", "deepseek_run_manifest.json"]
    produced = [EVAL / n for n in pats if (EVAL / n).exists()]
    if not produced:
        skip("nothing found in data/eval/ - did generation run?",
             produced_by="Step 5")
    else:
        for f in produced:
            print("      %-34s %s" % (f.name, fmt_bytes(f.stat().st_size)))

        EXPORT = ROOT.parent / "generated_output"
        EXPORT.mkdir(parents=True, exist_ok=True)
        copied = []
        for f in produced:
            shutil.copy2(f, EXPORT / f.name); copied.append(EXPORT / f.name)
        print("")
        print("   Copied %d file(s) to:" % len(copied))
        print("      %s" % EXPORT.resolve())
        check(len(copied) > 0, "exported %d file(s)" % len(copied))

        zip_path = EXPORT / ("llm_generated_%s.zip" % RUN_ID)
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for f in produced:
                z.write(f, f.name)
        print("")
        print("   Zipped -> %s (%s)" % (zip_path.name, fmt_bytes(zip_path.stat().st_size)))

        if IS_COLAB:
            try:
                from google.colab import files
                files.download(str(zip_path))
                note("Download started - check your browser's downloads.")
            except Exception as e:
                note("Auto-download failed (%s); fetch %s from the Files pane."
                     % (e, zip_path.name))
        else:
            print("")
            print("   Local run: your CSVs are in the folder above. Open")
            print("   generated_output/ - nothing to download.")
        rec["export_dir"] = str(EXPORT.resolve())